In [38]:
# ==========================================================
# CELL 1 — IMPORTS & CONFIGURATION
# ==========================================================

import os
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import mne
import torch

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

from torch.utils.data import (
    TensorDataset,
    DataLoader
)

from braindecode.models import ShallowFBCSPNet

warnings.filterwarnings("ignore")

# ==========================================================
# PATHS
# ==========================================================

PROJECT_DIR = "/Users/prajwalnara/Documents/EEG-motor-imagery"

RAW_DATA_DIR = os.path.join(
    PROJECT_DIR,
    "data/raw/physionet.org/files/eegmmidb/1.0.0"
)

MODEL_PATH = os.path.join(
    PROJECT_DIR,
    "models/shallow_convnet/50_subjects/best_model.pth"
)

NORMALIZATION_DIR = os.path.join(
    PROJECT_DIR,
    "data/shallow_convnet_splits/50_subjects"
)

RESULT_DIR = os.path.join(
    PROJECT_DIR,
    "results/real_movement/shallowconvnet/50_subjects"
)

os.makedirs(
    RESULT_DIR,
    exist_ok=True
)

# ==========================================================
# EXPERIMENT SETTINGS
# ==========================================================

TEST_SUBJECTS = list(range(43,51))

RUNS = [3,7,11]

EXPECTED_TIMEPOINTS = 641

SFREQ = 160

DEVICE = torch.device(
    "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)

print(f"Device : {DEVICE}")
print(f"Subjects : {TEST_SUBJECTS}")
print(f"Runs : {RUNS}")

Device : mps
Subjects : [43, 44, 45, 46, 47, 48, 49, 50]
Runs : [3, 7, 11]


In [39]:
# ==========================================================
# CELL 2 — LOAD TRAINED SHALLOWCONVNET
# ==========================================================

checkpoint = torch.load(
    MODEL_PATH,
    map_location=DEVICE
)

print("Checkpoint loaded successfully.\n")

print("Checkpoint Contents:")
for key in checkpoint.keys():
    print(f"  {key}")

print()

# ----------------------------------------------------------
# Recreate model
# ----------------------------------------------------------

model = ShallowFBCSPNet(

    n_chans=checkpoint["n_channels"],

    n_outputs=3,

    n_times=checkpoint["n_times"],

    sfreq=checkpoint["sfreq"]

)

# ----------------------------------------------------------
# Load trained weights
# ----------------------------------------------------------

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model = model.to(DEVICE)

model.eval()

print("=" * 60)
print("Model Ready")
print("=" * 60)

print(f"Best Epoch           : {checkpoint['best_epoch']}")
print(f"Validation Accuracy  : {checkpoint['best_val_accuracy']:.4f}")
print(f"Device               : {DEVICE}")

Checkpoint loaded successfully.

Checkpoint Contents:
  model_state_dict
  best_epoch
  best_val_accuracy
  n_channels
  n_times
  n_classes
  sfreq

Model Ready
Best Epoch           : 75
Validation Accuracy  : 0.6361
Device               : mps


In [41]:
# ==========================================================
# CELL 3 — LOAD TRAINING NORMALIZATION
# ==========================================================

channel_mean = np.load(
    os.path.join(
        NORMALIZATION_DIR,
        "channel_mean.npy"
    )
)

channel_std = np.load(
    os.path.join(
        NORMALIZATION_DIR,
        "channel_std.npy"
    )
)

print("=" * 60)
print("Normalization Statistics")
print("=" * 60)

print(f"Mean Shape : {channel_mean.shape}")
print(f"Std Shape  : {channel_std.shape}")

print()
print("Mean Statistics")
print(f"Mean : {channel_mean.mean():.6e}")

print()
print("Std Statistics")
print(f"Mean Std : {channel_std.mean():.6e}")

Normalization Statistics
Mean Shape : (1, 64, 1)
Std Shape  : (1, 64, 1)

Mean Statistics
Mean : 2.767720e-09

Std Statistics
Mean Std : 2.520006e-05


In [48]:
# ==========================================================
# CELL 4 — PROCESS ALL TEST SUBJECTS
# ==========================================================

import gc

all_true_labels = []
all_predictions = []
all_probabilities = []

subject_accuracy = {}
prediction_records = []

label_map = {
    "T0": 0,
    "T1": 1,
    "T2": 2
}

for subject in TEST_SUBJECTS:

    subject_id = f"S{subject:03d}"

    print("\n" + "=" * 70)
    print(f"Processing {subject_id}")
    print("=" * 70)

    X_runs = []
    y_runs = []

    # ------------------------------------------------------
    # Process R03, R07, R11
    # ------------------------------------------------------

    for run in RUNS:

        print(f"Loading {subject_id}R{run:02d}")

        file_path = os.path.join(
            RAW_DATA_DIR,
            subject_id,
            f"{subject_id}R{run:02d}.edf"
        )

        raw = mne.io.read_raw_edf(
            file_path,
            preload=True,
            verbose=False
        )

        # Bandpass filter
        raw.filter(
            l_freq=4,
            h_freq=40,
            fir_design="firwin",
            verbose=False
        )

        # Events
        events, event_id = mne.events_from_annotations(
            raw,
            verbose=False
        )

        event_mapping = {
            "rest": event_id["T0"],
            "left": event_id["T1"],
            "right": event_id["T2"]
        }

        epochs = mne.Epochs(
            raw,
            events,
            event_id=event_mapping,
            tmin=0,
            tmax=4,
            baseline=None,
            preload=True,
            verbose=False
        )

        if len(epochs.times) != EXPECTED_TIMEPOINTS:
            raise ValueError(
                f"{subject_id} R{run:02d} has "
                f"{len(epochs.times)} timepoints."
            )

        X = epochs.get_data(copy=True)

        labels = epochs.events[:, -1]

        y = np.array([
            0 if l == event_mapping["rest"]
            else 1 if l == event_mapping["left"]
            else 2
            for l in labels
        ])

        X_runs.append(X)
        y_runs.append(y)

    # ------------------------------------------------------
    # Combine Runs
    # ------------------------------------------------------

    X_subject = np.concatenate(X_runs, axis=0)
    y_subject = np.concatenate(y_runs, axis=0)

    print(f"Shape : {X_subject.shape}")

    # ------------------------------------------------------
    # Normalize
    # ------------------------------------------------------

    X_subject = (X_subject - channel_mean) / channel_std

    # ------------------------------------------------------
    # Tensor
    # ------------------------------------------------------

    X_tensor = torch.tensor(
        X_subject,
        dtype=torch.float32
    )

    y_tensor = torch.tensor(
        y_subject,
        dtype=torch.long
    )

    test_loader = DataLoader(
        TensorDataset(X_tensor, y_tensor),
        batch_size=32,
        shuffle=False
    )

    # ------------------------------------------------------
    # Inference
    # ------------------------------------------------------

    model.eval()

    subject_predictions = []
    subject_probabilities = []

    with torch.no_grad():

        for X_batch, _ in test_loader:

            X_batch = X_batch.to(DEVICE)

            outputs = model(X_batch)

            probs = torch.softmax(outputs, dim=1)

            preds = torch.argmax(probs, dim=1)

            subject_predictions.extend(
                preds.cpu().numpy()
            )

            subject_probabilities.extend(
                probs.cpu().numpy()
            )

    subject_predictions = np.array(subject_predictions)
    subject_probabilities = np.array(subject_probabilities)

    acc = accuracy_score(
        y_subject,
        subject_predictions
    )

    subject_accuracy[subject_id] = float(acc)

    print(f"Accuracy : {acc:.4f}")

    # ------------------------------------------------------
    # Store Overall Results
    # ------------------------------------------------------

    all_true_labels.extend(y_subject.tolist())
    all_predictions.extend(subject_predictions.tolist())
    all_probabilities.extend(subject_probabilities.tolist())

    # ------------------------------------------------------
    # Store Every Trial
    # ------------------------------------------------------

    for i in range(len(y_subject)):

        prediction_records.append({

            "Subject": subject_id,
            "Trial": i + 1,
            "True": int(y_subject[i]),
            "Prediction": int(subject_predictions[i]),
            "Rest_Prob": float(subject_probabilities[i][0]),
            "Left_Prob": float(subject_probabilities[i][1]),
            "Right_Prob": float(subject_probabilities[i][2])

        })

    # ------------------------------------------------------
    # Memory Cleanup
    # ------------------------------------------------------

    del raw
    del epochs

    del X_subject
    del y_subject

    del X_tensor
    del y_tensor

    del X_runs
    del y_runs

    del test_loader

    gc.collect()

    if torch.backends.mps.is_available():
        torch.mps.empty_cache()

    print(f"Completed {subject_id}")


Processing S043
Loading S043R03
Loading S043R07
Loading S043R11
Shape : (90, 64, 641)
Accuracy : 0.6000
Completed S043

Processing S044
Loading S044R03
Loading S044R07
Loading S044R11
Shape : (90, 64, 641)
Accuracy : 0.6444
Completed S044

Processing S045
Loading S045R03
Loading S045R07
Loading S045R11
Shape : (90, 64, 641)
Accuracy : 0.7222
Completed S045

Processing S046
Loading S046R03
Loading S046R07
Loading S046R11
Shape : (90, 64, 641)
Accuracy : 0.5556
Completed S046

Processing S047
Loading S047R03
Loading S047R07
Loading S047R11
Shape : (90, 64, 641)
Accuracy : 0.5889
Completed S047

Processing S048
Loading S048R03
Loading S048R07
Loading S048R11
Shape : (90, 64, 641)
Accuracy : 0.6222
Completed S048

Processing S049
Loading S049R03
Loading S049R07
Loading S049R11
Shape : (90, 64, 641)
Accuracy : 0.7556
Completed S049

Processing S050
Loading S050R03
Loading S050R07
Loading S050R11
Shape : (90, 64, 641)
Accuracy : 0.6000
Completed S050


In [49]:
overall_accuracy = accuracy_score(
    all_true_labels,
    all_predictions
)

report = classification_report(
    all_true_labels,
    all_predictions,
    target_names=["Rest", "Left", "Right"],
    digits=4
)

confusion = confusion_matrix(
    all_true_labels,
    all_predictions
)

print("=" * 70)
print("FINAL RESULTS")
print("=" * 70)

print(f"\nOverall Accuracy : {overall_accuracy:.4f}\n")

print(report)

print("\nConfusion Matrix\n")

print(confusion)

print("\nSubject-wise Accuracy")

for subject, acc in subject_accuracy.items():
    print(f"{subject}: {acc:.4f}")

FINAL RESULTS

Overall Accuracy : 0.6361

              precision    recall  f1-score   support

        Rest     0.6537    0.8861    0.7524       360
        Left     0.5862    0.4722    0.5231       180
       Right     0.6207    0.3000    0.4045       180

    accuracy                         0.6361       720
   macro avg     0.6202    0.5528    0.5600       720
weighted avg     0.6286    0.6361    0.6081       720


Confusion Matrix

[[319  15  26]
 [ 88  85   7]
 [ 81  45  54]]

Subject-wise Accuracy
S043: 0.6000
S044: 0.6444
S045: 0.7222
S046: 0.5556
S047: 0.5889
S048: 0.6222
S049: 0.7556
S050: 0.6000


In [50]:
# ==========================================================
# CELL 6 — SAVE RESULTS
# ==========================================================

os.makedirs(
    RESULT_DIR,
    exist_ok=True
)

# ----------------------------------------------------------
# Save Classification Report
# ----------------------------------------------------------

report_path = os.path.join(
    RESULT_DIR,
    "classification_report.txt"
)

with open(report_path, "w") as f:

    f.write(report)

# ----------------------------------------------------------
# Save Confusion Matrix
# ----------------------------------------------------------

confusion_path = os.path.join(
    RESULT_DIR,
    "confusion_matrix.npy"
)

np.save(
    confusion_path,
    confusion
)

# ----------------------------------------------------------
# Save Overall Metrics
# ----------------------------------------------------------

metrics = {

    "overall_accuracy": float(overall_accuracy),

    "total_subjects": len(TEST_SUBJECTS),

    "total_trials": len(all_true_labels)

}

metrics_path = os.path.join(
    RESULT_DIR,
    "overall_metrics.json"
)

with open(metrics_path, "w") as f:

    json.dump(
        metrics,
        f,
        indent=4
    )

# ----------------------------------------------------------
# Save Subject-wise Accuracy
# ----------------------------------------------------------

subject_accuracy_path = os.path.join(
    RESULT_DIR,
    "subject_accuracy.json"
)

with open(subject_accuracy_path, "w") as f:

    json.dump(
        subject_accuracy,
        f,
        indent=4
    )

# ----------------------------------------------------------
# Save Prediction Details
# ----------------------------------------------------------

prediction_df = pd.DataFrame(
    prediction_records
)

prediction_csv_path = os.path.join(
    RESULT_DIR,
    "predictions.csv"
)

prediction_df.to_csv(
    prediction_csv_path,
    index=False
)

# ----------------------------------------------------------
# Save Prediction Probabilities
# ----------------------------------------------------------

probability_path = os.path.join(
    RESULT_DIR,
    "prediction_probabilities.npy"
)

np.save(
    probability_path,
    np.array(all_probabilities)
)

# ----------------------------------------------------------
# Save Readable Summary
# ----------------------------------------------------------

summary_path = os.path.join(
    RESULT_DIR,
    "summary.txt"
)

with open(summary_path, "w") as f:

    f.write("=" * 60 + "\n")
    f.write("REAL MOVEMENT EVALUATION\n")
    f.write("=" * 60 + "\n\n")

    f.write(f"Model              : ShallowConvNet\n")
    f.write(f"Training Dataset   : Motor Imagery (R04,R08,R12)\n")
    f.write(f"Testing Dataset    : Motor Execution (R03,R07,R11)\n")
    f.write(f"Subjects           : {TEST_SUBJECTS}\n")
    f.write(f"Overall Accuracy   : {overall_accuracy:.4f}\n")
    f.write(f"Total Trials       : {len(all_true_labels)}\n")
    f.write(f"Total Subjects     : {len(TEST_SUBJECTS)}\n")

# ----------------------------------------------------------
# Display Summary
# ----------------------------------------------------------

print("\n" + "=" * 70)
print("RESULTS SAVED SUCCESSFULLY")
print("=" * 70)

print(f"\nLocation:\n{RESULT_DIR}\n")

print("Saved Files")
print("-" * 70)

print("✓ classification_report.txt")
print("✓ confusion_matrix.npy")
print("✓ overall_metrics.json")
print("✓ subject_accuracy.json")
print("✓ predictions.csv")
print("✓ prediction_probabilities.npy")
print("✓ summary.txt")

print("\nOverall Accuracy :", f"{overall_accuracy:.4f}")
print("=" * 70)


RESULTS SAVED SUCCESSFULLY

Location:
/Users/prajwalnara/Documents/EEG-motor-imagery/results/real_movement/shallowconvnet/50_subjects

Saved Files
----------------------------------------------------------------------
✓ classification_report.txt
✓ confusion_matrix.npy
✓ overall_metrics.json
✓ subject_accuracy.json
✓ predictions.csv
✓ prediction_probabilities.npy
✓ summary.txt

Overall Accuracy : 0.6361
